In [28]:
import pickle as pkl
model=pkl.dump(model,open("model.pkl","wb"))
labelEncoder=pkl.dump(labelEncoder,open("labelEncoder.pkl",'wb'))

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
import xgboost
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, precision_score,f1_score,recall_score,classification_report

In [7]:
# dataset
loan=pd.read_csv("loan_approval_training_dataset.csv")

In [ ]:
loan

In [ ]:
# describe
loan.describe()


In [8]:
# handle spaces and other syymbols to columns
loan.columns = loan.columns.str.strip().str.replace(" ", "").str.replace("_", "")
loan.columns = loan.columns.str[0].str.lower() + loan.columns.str[1:]

In [9]:
# Handling spaces for records
for records in loan.columns:
    if loan[records].dtype not in ['int64','float64']:
        loan[records]=loan[records].str.strip()

In [ ]:
loan

In [10]:
# exclude loadid,cibilscore and loanterm cols
loan=loan.drop(columns=['loanid','cibilscore','loanterm'])

In [11]:

# encode education and selfemployed columns
loan['education'] = loan['education'].map({"Graduate": 1, "Not Graduate": 0})
loan['selfemployed'] = loan['selfemployed'].map({"Yes": 1, "No": 0})

In [ ]:
loan

In [ ]:
# check null values
loan.isnull().sum()
    

In [12]:
# extract num and non-num cols
colNum=[cols for cols in loan.columns if loan[cols].dtype!="object"]
# colNum
colText=[columns for columns in loan.columns if loan[columns].dtype not in ["int64","float64"]]
colText

['loanstatus']

In [ ]:
# loan['education'] = loan['education'].map({"Graduate": 1, "Not Graduate": 0})

In [ ]:
print(loan['education'].unique())
print(loan['education'].isna().sum())
print(loan.shape)

In [ ]:
loan

In [13]:
# /handle outliers
for handleOutliers in loan.columns:
    if handleOutliers not in colText:
        Q1=loan[handleOutliers].quantile(0.25)
        Q3=loan[handleOutliers].quantile(0.75)
        IQR=Q3-Q1
    
        Upper=Q3+1.5*IQR
        Lower=Q1-1.5*IQR
        
        countOut=loan.loc[(loan[handleOutliers]<Lower) | (loan[handleOutliers]>Upper)].shape[0]
        if countOut!=0:
            loan[handleOutliers]=loan[handleOutliers].clip(lower=Lower,upper=Upper)

In [ ]:
loan['loanstatus']

In [14]:
# Split data
loan['loanstatus']=loan['loanstatus'].str.strip()
x=loan.drop(columns=["loanstatus"])
y=loan["loanstatus"]
xTrain,xTest,yTrain,yTest=train_test_split(x,y,test_size=0.25,random_state=45)

In [ ]:
loan

In [15]:
# Encoder for target var
labelEncoder=LabelEncoder()

In [16]:
# encode yTrain
yTrain=labelEncoder.fit_transform(yTrain)

In [17]:
# encode y test
yTest=labelEncoder.transform(yTest)

In [ ]:
xTrain

In [ ]:
yTrain

In [ ]:
xTrain

In [ ]:
yTrain

In [ ]:
yTrain=pd.DataFrame(yTrain)

In [ ]:
yTrain.columns=['loanstatus']

In [ ]:
# check corralation
xTrain.corrwith(yTrain['loanstatus'])

In [ ]:
trainingFrame=pd.concat([xTrain,yTrain],axis=1)

In [ ]:
trainingFrame

In [ ]:
# Corr with heat Map
plt.figure(figsize=(20,5))
sns.heatmap(trainingFrame.corr(),annot=True)

In [18]:
# Realationship is non linear, xgboost is a go to Model
model=XGBClassifier(learning_rate=0.1, reg_alpha=0.1,reg_lambda=0.1)
model.fit(xTrain,yTrain)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [19]:
# Prediction
prediction=model.predict(xTest)

In [20]:
# Compute classification report
modelReport=classification_report(yTest,prediction,target_names=labelEncoder.classes_,output_dict=True)

In [21]:
modelReport

{'Approved': {'precision': 0.6262740656851642,
  'recall': 0.8417047184170472,
  'f1-score': 0.7181818181818181,
  'support': 657.0},
 'Rejected': {'precision': 0.43783783783783786,
  'recall': 0.19708029197080293,
  'f1-score': 0.27181208053691275,
  'support': 411.0},
 'accuracy': 0.5936329588014981,
 'macro avg': {'precision': 0.532055951761501,
  'recall': 0.5193925051939251,
  'f1-score': 0.49499694935936545,
  'support': 1068.0},
 'weighted avg': {'precision': 0.5537578768787492,
  'recall': 0.5936329588014981,
  'f1-score': 0.5464047000431889,
  'support': 1068.0}}

In [22]:
reportFrame=pd.DataFrame(modelReport).transpose()

In [23]:
reportFrame

,precision,recall,f1-score,support
Approved,0.626274,0.841705,0.718182,657.000000
Rejected,0.437838,0.197080,0.271812,411.000000
accuracy,0.593633,0.593633,0.593633,0.593633
macro avg,0.532056,0.519393,0.494997,1068.000000
weighted avg,0.553758,0.593633,0.546405,1068.000000


In [24]:
# computer probability
probability=model.predict_proba(xTest)

In [25]:
model.classes_

array([0, 1])

In [ ]:
probability

In [26]:
probaFrame=pd.DataFrame(probability,columns=labelEncoder.classes_)

In [27]:
probaFrame

,Approved,Rejected
0,0.406492,0.593508
1,0.622618,0.377382
2,0.640738,0.359262
3,0.734605,0.265395
4,0.505649,0.494351
...,...,...
1063,0.884067,0.115933
1064,0.604279,0.395721
1065,0.739825,0.260175
1066,0.734264,0.265736
